# Preprocessing, Normalization and Data Augmentation

In [10]:
import os
import shutil
import numpy as np
import torch
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset
import torch.nn as nn
from torchvision import models
import torch.optim as optim
from torch.utils.data import DataLoader
# Import metrics from scikit-learn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr

Define the transformation pipeline for the TRAINING data.
This includes augmentation, preprocessing (resizing), and normalization.

In [11]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3),
    transforms.ToTensor(), # Converts to tensor and scales to [0, 1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Normalizes
])

Define the transformation pipeline for VALIDATION & TEST data.
This only includes essential preprocessing and normalization, NO augmentation.

In [12]:
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Process and save data

In [13]:
def process_and_save(ids, image_dir, annotation_dir, output_dir, transform, is_train=False, augment_factor=5):
    """
    Processes a list of image IDs and saves the transformed images and annotations.
    If is_train is True, it creates multiple augmented versions.
    """
    # Create subdirectories for images and annotations
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'annotations'), exist_ok=True)
    
    print(f"Processing data for: {output_dir}")
    for img_id in tqdm(ids):
        # Find the full image file path (handles .jpg, .png, etc.)
        img_path = None
        for ext in ['.jpg', '.png', '.jpeg']:
            if os.path.exists(os.path.join(image_dir, img_id + ext)):
                img_path = os.path.join(image_dir, img_id + ext)
                break
        if not img_path: continue

        image = Image.open(img_path).convert('RGB')
        
        num_versions = augment_factor if is_train else 1
        
        for i in range(num_versions):
            # Apply the transformations to get the final processed tensor
            image_tensor = transform(image)
            
            new_filename_base = f"{img_id}_aug_{i}" if is_train else img_id
            
            # Save the processed image tensor
            torch.save(image_tensor, os.path.join(output_dir, 'images', f"{new_filename_base}.pt"))
            
            # Copy the corresponding annotation files, renaming them to match
            for ann_type in ['exp', 'val', 'aro', 'lnd']:
                src_ann_path = os.path.join(annotation_dir, f"{img_id}_{ann_type}.npy")
                if os.path.exists(src_ann_path):
                    dst_ann_path = os.path.join(output_dir, 'annotations', f"{new_filename_base}_{ann_type}.npy")
                    shutil.copyfile(src_ann_path, dst_ann_path)


In [ ]:
if __name__ == '__main__':

    RAW_IMAGE_DIR = '/kaggle/input/dataset/Dataset/Dataset/images'
    RAW_ANNOTATION_DIR = '/kaggle/input/dataset/Dataset/Dataset/annotations'
    PROCESSED_DATA_DIR = '/kaggle/input/processed-data/processed_data' # A new folder will be created here

    # --- Split Ratios ---
    VALIDATION_SIZE = 0.15
    TEST_SIZE = 0.15
    AUGMENTATION_FACTOR = 4 # Create 4 augmented versions for each training image

    all_image_ids = sorted([
        os.path.splitext(f)[0] for f in os.listdir(RAW_IMAGE_DIR) 
        if f.endswith(('.png', '.jpg', '.jpeg'))
    ])
    
    if not all_image_ids: raise ValueError(f"No images found in {RAW_IMAGE_DIR}.")

    # 1. Split IDs into train, validation, and test sets
    train_ids, temp_ids = train_test_split(all_image_ids, test_size=(VALIDATION_SIZE + TEST_SIZE), random_state=42)
    val_ids, test_ids = train_test_split(temp_ids, test_size=(TEST_SIZE / (VALIDATION_SIZE + TEST_SIZE)), random_state=42)

    print(f"Dataset Split:")
    print(f"  - Training samples:   {len(train_ids)} (will be augmented to ~{len(train_ids) * AUGMENTATION_FACTOR})")
    print(f"  - Validation samples: {len(val_ids)}")
    print(f"  - Test samples:       {len(test_ids)}")

    # 2. Process and save each dataset split
    process_and_save(train_ids, RAW_IMAGE_DIR, RAW_ANNOTATION_DIR, 
                     os.path.join(PROCESSED_DATA_DIR, 'train'),
                     train_transforms, is_train=True, augment_factor=AUGMENTATION_FACTOR)

    process_and_save(val_ids, RAW_IMAGE_DIR, RAW_ANNOTATION_DIR, 
                     os.path.join(PROCESSED_DATA_DIR, 'validation'),
                     val_test_transforms, is_train=False)

    process_and_save(test_ids, RAW_IMAGE_DIR, RAW_ANNOTATION_DIR, 
                     os.path.join(PROCESSED_DATA_DIR, 'test'),
                     val_test_transforms, is_train=False)

    print(f"\nData preparation complete. Processed data is saved in '{PROCESSED_DATA_DIR}'.")


# DataLoader

In [17]:
class FacialExpressionDataset(Dataset):
    """
    Custom PyTorch Dataset for loading pre-processed facial expression data.
    It expects a directory containing 'images' and 'annotations' subfolders
    where data has been saved by your preprocessing script.
    """
    def __init__(self, data_dir):
        """
        Args:
            data_dir (string): Directory for a specific data split (e.g., 'processed_data/train').
        """
        self.image_dir = os.path.join(data_dir, 'images')
        self.annotation_dir = os.path.join(data_dir, 'annotations')

        if not os.path.isdir(self.image_dir):
            raise FileNotFoundError(f"Processed image directory not found: {self.image_dir}")

        # Get a sorted list of unique sample IDs from the image filenames
        # e.g., ['image_0001', 'image_0002_aug_0', 'image_0002_aug_1', ...]
        self.sample_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(self.image_dir) if f.endswith('.pt')])

        print(f"Initialized dataset from '{data_dir}'. Found {len(self.sample_ids)} samples.")

    def __len__(self):
        """Returns the total number of samples in the dataset."""
        return len(self.sample_ids)

    def __getitem__(self, idx):
        """
        Fetches the sample (image and its labels) at the given index.

        Args:
            idx (int): The index of the sample to fetch.

        Returns:
            tuple: A tuple containing (image, labels_dict).
                   'image' is the image tensor.
                   'labels_dict' is a dictionary with 'expression', 'valence', and 'arousal' tensors.
        """
        # Get the unique ID for the sample at the requested index
        sample_id = self.sample_ids[idx]
        
        # --- 1. Load Image Tensor ---
        image_path = os.path.join(self.image_dir, f"{sample_id}.pt")
        image = torch.load(image_path)

        # --- 2. Load Annotations ---
        # Construct paths to the corresponding annotation files
        exp_path = os.path.join(self.annotation_dir, f"{sample_id}_exp.npy")
        val_path = os.path.join(self.annotation_dir, f"{sample_id}_val.npy")
        aro_path = os.path.join(self.annotation_dir, f"{sample_id}_aro.npy")

        # Load the numpy arrays and extract the scalar value using .item()
        # This fixes the error from your original notebook.
        expression = np.load(exp_path).item()
        valence = np.load(val_path).item()
        arousal = np.load(aro_path).item()
        
        # --- 3. Package Labels into a Dictionary ---
        labels = {
            'expression': torch.tensor(int(expression), dtype=torch.long),  # Ensure integer
            'valence': torch.tensor(float(valence), dtype=torch.float32),
            'arousal': torch.tensor(float(arousal), dtype=torch.float32)
        }

        return image, labels

In [18]:
class MultiTaskResNet50(nn.Module):
    """
    A pre-trained ResNet50 model adapted for multi-task learning.
    It has three distinct output heads to predict:
    1. Facial Expression (Classification)
    2. Valence (Regression)
    3. Arousal (Regression)
    """
    def __init__(self, num_expression_classes=8):
        super(MultiTaskResNet50, self).__init__()
        
        # Load a ResNet50 model pre-trained on ImageNet
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        
        # Get the number of input features for the classifier
        num_ftrs = self.resnet.fc.in_features
        
        # Replace the final fully-connected layer with an identity layer.
        # This allows us to get the feature vector from the convolutional base.
        self.resnet.fc = nn.Identity()

        # --- Define the three separate output heads ---
        
        # 1. Expression classification head
        self.expression_head = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_expression_classes)
        )
        
        # 2. Valence regression head
        self.valence_head = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1) # Outputs a single continuous value
        )

        # 3. Arousal regression head
        self.arousal_head = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1) # Outputs a single continuous value
        )

    def forward(self, x):
        """Defines the forward pass of the model."""
        # 1. Get the base features from the ResNet backbone
        features = self.resnet(x)
        
        # 2. Pass the features through each independent head
        expression_output = self.expression_head(features)
        valence_output = self.valence_head(features)
        arousal_output = self.arousal_head(features)
        
        # 3. Return the outputs in a dictionary for clarity
        outputs = {
            'expression': expression_output,
            'valence': valence_output.squeeze(1), # Remove last dim: [B, 1] -> [B]
            'arousal': arousal_output.squeeze(1)  # Remove last dim: [B, 1] -> [B]
        }
        
        return outputs

In [19]:
# --- Configuration Section ---
PROCESSED_DATA_DIR = '/kaggle/input/processed-data/processed_data'

BATCH_SIZE = 32

LEARNING_RATE = 1e-4

NUM_EPOCHS = 15 # You can increase this for better performance

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

def main():
    # --- 1. Create Datasets and DataLoaders ---
    # This now uses our clean and efficient Dataset class
    train_dataset = FacialExpressionDataset(os.path.join(PROCESSED_DATA_DIR, 'train'))
    val_dataset = FacialExpressionDataset(os.path.join(PROCESSED_DATA_DIR, 'validation'))
    # The DataLoader handles batching, shuffling, and can use multiple CPU cores
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    # --- 2. Initialize Model, Loss Functions, and Optimizer ---
    model = MultiTaskResNet50(num_expression_classes=8).to(DEVICE)

    # Define separate loss functions for each task
    criterion_expression = nn.CrossEntropyLoss()      # For multi-class classification
    criterion_valence_arousal = nn.MSELoss()          # For regression (predicting continuous values)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # --- 3. Training and Validation Loop ---
    for epoch in range(NUM_EPOCHS):
        print(f"\\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")

        # --- Training Phase ---
        model.train()
        total_train_loss = 0.0

        for images, labels in tqdm(train_loader, desc="Training"):
            images = images.to(DEVICE)
            exp_labels = labels['expression'].to(DEVICE)
            val_labels = labels['valence'].to(DEVICE)
            aro_labels = labels['arousal'].to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)

            # Calculate loss for each task
            loss_exp = criterion_expression(outputs['expression'], exp_labels)
            loss_val = criterion_valence_arousal(outputs['valence'], val_labels)
            loss_aro = criterion_valence_arousal(outputs['arousal'], aro_labels)

            # Combine losses. A simple sum is a good starting point.
            # You could also weight them, e.g., total_loss = 1.5 * loss_exp + loss_val + loss_aro
            total_loss = loss_exp + loss_val + loss_aro

            total_loss.backward()
            optimizer.step()

            total_train_loss += total_loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        print(f"Average Training Loss: {avg_train_loss:.4f}")

        # --- Validation Phase ---
        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation"):
                images = images.to(DEVICE)
                exp_labels = labels['expression'].to(DEVICE)
                val_labels = labels['valence'].to(DEVICE)
                aro_labels = labels['arousal'].to(DEVICE)

                outputs = model(images)

                loss_exp = criterion_expression(outputs['expression'], exp_labels)
                loss_val = criterion_valence_arousal(outputs['valence'], val_labels)
                loss_aro = criterion_valence_arousal(outputs['arousal'], aro_labels)

                total_loss = loss_exp + loss_val + loss_aro
                total_val_loss += total_loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        print(f"Average Validation Loss: {avg_val_loss:.4f}")

    print("\\nTraining complete.")
    # Save the final model state
    torch.save(model.state_dict(), 'multitask_resnet50_final.pth')
    print("Model saved to multitask_resnet50_final.pth")
    

main()

Using device: cuda
Initialized dataset from '/kaggle/input/processed-data/processed_data/train'. Found 11196 samples.
Initialized dataset from '/kaggle/input/processed-data/processed_data/validation'. Found 600 samples.
\n--- Epoch 1/15 ---


Training: 100%|██████████| 350/350 [03:25<00:00,  1.70it/s]


Average Training Loss: 1.9233


Validation: 100%|██████████| 19/19 [00:06<00:00,  3.12it/s]


Average Validation Loss: 1.8933
\n--- Epoch 2/15 ---


...

Average Validation Loss: 3.2666
\n--- Epoch 15/15 ---


Training: 100%|██████████| 350/350 [02:47<00:00,  2.09it/s]


Average Training Loss: 0.0496


Validation: 100%|██████████| 19/19 [00:03<00:00,  4.77it/s]


Average Validation Loss: 3.3210
\nTraining complete.
Model saved to multitask_resnet50_final.pth


# Evaluation

In [21]:
# --- Configuration ---
PROCESSED_DATA_DIR = '/kaggle/input/processed-data/processed_data'
MODEL_PATH = 'multitask_resnet50_final.pth' # Path to your saved model
BATCH_SIZE = 32
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {DEVICE}")

# --- Custom Metric Implementations ---

def sign_agreement_metric(y_true, y_pred):
    """
    Calculates the Sign Agreement Metric (SAGR).
    Penalizes incorrect sign alongside deviation from value.
    """
    sagr = np.mean(np.sign(y_true) == np.sign(y_pred)) * 100
    # A more complete implementation could also factor in the magnitude of error,
    # but based on the description, a simple sign agreement percentage is a good start.
    # The description implies SAGR is a component, not the final metric. Let's use a simple definition.
    # For a more robust SAGR:
    # error = np.abs(y_true - y_pred)
    # sign_match = (np.sign(y_true) == np.sign(y_pred)).astype(float)
    # sagr_score = np.mean(sign_match * (1 - error))
    return sagr

def concordance_correlation_coefficient(y_true, y_pred):
    """
    Calculates the Concordance Correlation Coefficient (CCC).
    """
    # Pearson's correlation coefficient
    cor = np.corrcoef(y_true, y_pred)[0, 1]
    
    # Mean and variance of true and predicted values
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    
    # Calculate CCC
    sd_true = np.std(y_true)
    sd_pred = np.std(y_pred)
    numerator = 2 * cor * sd_true * sd_pred
    denominator = var_true + var_pred + (mean_true - mean_pred)**2
    
    return numerator / denominator

def main():
    # --- 1. Load Test Dataset ---
    test_dataset = FacialExpressionDataset(os.path.join(PROCESSED_DATA_DIR, 'test'))
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # --- 2. Load Trained Model ---
    model = MultiTaskResNet50(num_expression_classes=8)
    # Load the state dict, ensuring it's mapped to the correct device (important if trained on GPU, evaluated on CPU)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device(DEVICE)))
    model.to(DEVICE)
    model.eval() # Set the model to evaluation mode

    # --- 3. Get Predictions for the Entire Test Set ---
    all_exp_preds, all_exp_labels = [], []
    all_val_preds, all_val_labels = [], []
    all_aro_preds, all_aro_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating on Test Set"):
            images = images.to(DEVICE)
            
            # Forward pass
            outputs = model(images)
            
            # For expression, get the predicted class index
            exp_preds = torch.argmax(outputs['expression'], dim=1)
            all_exp_preds.extend(exp_preds.cpu().numpy())
            all_exp_labels.extend(labels['expression'].cpu().numpy())

            # For valence and arousal, get the raw regression output
            all_val_preds.extend(outputs['valence'].cpu().numpy())
            all_val_labels.extend(labels['valence'].cpu().numpy())
            
            all_aro_preds.extend(outputs['arousal'].cpu().numpy())
            all_aro_labels.extend(labels['arousal'].cpu().numpy())

    # Convert lists to numpy arrays for metric calculations
    all_exp_preds = np.array(all_exp_preds)
    all_exp_labels = np.array(all_exp_labels)
    all_val_preds = np.array(all_val_preds)
    all_val_labels = np.array(all_val_labels)
    all_aro_preds = np.array(all_aro_preds)
    all_aro_labels = np.array(all_aro_labels)

    # --- 4. Calculate and Print Metrics ---

    print("\n" + "="*50)
    print("      EVALUATION RESULTS ON TEST SET")
    print("="*50 + "\n")

    # === Categorical Classification Metrics (Expression) ===
    print("--- Facial Expression (Categorical) ---")
    accuracy = accuracy_score(all_exp_labels, all_exp_preds)
    # Use 'weighted' for F1-score to account for class imbalance
    f1 = f1_score(all_exp_labels, all_exp_preds, average='weighted') 
    kappa = cohen_kappa_score(all_exp_labels, all_exp_preds)
    
    print(f"  Accuracy:       {accuracy:.4f}")
    print(f"  F1-Score:       {f1:.4f}")
    print(f"  Cohen's Kappa:  {kappa:.4f}")
    # Note: AUC and AUC-PR are more complex for multi-class and often not reported as a single number.
    # We are omitting them for this direct evaluation as per common practice, but they can be
    # calculated using one-vs-rest strategies if strictly required.
    print("\n")

    # === Continuous Domain Metrics (Valence) ===
    print("--- Valence (Continuous) ---")
    rmse_val = np.sqrt(mean_squared_error(all_val_labels, all_val_preds))
    corr_val, _ = pearsonr(all_val_labels, all_val_preds)
    sagr_val = sign_agreement_metric(all_val_labels, all_val_preds)
    ccc_val = concordance_correlation_coefficient(all_val_labels, all_val_preds)

    print(f"  Root Mean Square Error (RMSE): {rmse_val:.4f}")
    print(f"  Pearson Correlation (CORR):    {corr_val:.4f}")
    print(f"  Sign Agreement (SAGR):         {sagr_val:.2f}%")
    print(f"  Concordance Corr Coeff (CCC):  {ccc_val:.4f}")
    print("\n")

    # === Continuous Domain Metrics (Arousal) ===
    print("--- Arousal (Continuous) ---")
    rmse_aro = np.sqrt(mean_squared_error(all_aro_labels, all_aro_preds))
    corr_aro, _ = pearsonr(all_aro_labels, all_aro_preds)
    sagr_aro = sign_agreement_metric(all_aro_labels, all_aro_preds)
    ccc_aro = concordance_correlation_coefficient(all_aro_labels, all_aro_preds)

    print(f"  Root Mean Square Error (RMSE): {rmse_aro:.4f}")
    print(f"  Pearson Correlation (CORR):    {corr_aro:.4f}")
    print(f"  Sign Agreement (SAGR):         {sagr_aro:.2f}%")
    print(f"  Concordance Corr Coeff (CCC):  {ccc_aro:.4f}")
    print("\n" + "="*50)


if __name__ == '__main__':
    main()

Using device: cuda
Initialized dataset from '/kaggle/input/processed-data/processed_data/test'. Found 600 samples.


Evaluating on Test Set: 100%|██████████| 19/19 [00:23<00:00,  1.24s/it]


      EVALUATION RESULTS ON TEST SET

--- Facial Expression (Categorical) ---
  Accuracy:       0.4417
  F1-Score:       0.4424
  Cohen's Kappa:  0.3620


--- Valence (Continuous) ---
  Root Mean Square Error (RMSE): 0.4227
  Pearson Correlation (CORR):    0.5150
  Sign Agreement (SAGR):         76.00%
  Concordance Corr Coeff (CCC):  0.5075


--- Arousal (Continuous) ---
  Root Mean Square Error (RMSE): 0.3696
  Pearson Correlation (CORR):    0.4078
  Sign Agreement (SAGR):         76.50%
  Concordance Corr Coeff (CCC):  0.3836



# DenseNet121

In [26]:
class MultiTaskDenseNet121(nn.Module):
    """
    A pre-trained DenseNet121 model adapted for our multi-task problem.
    This architecture is known for its feature reuse, making it very parameter-efficient.
    """
    def __init__(self, num_expression_classes=8):
        super(MultiTaskDenseNet121, self).__init__()
        
        # Load a DenseNet121 model pre-trained on ImageNet
        self.densenet = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        
        # Get the number of input features for the classifier layer
        num_ftrs = self.densenet.classifier.in_features
        
        # Replace the final classifier with an identity layer to extract features
        self.densenet.classifier = nn.Identity()

        # --- Define the three separate output heads ---
        
        # 1. Expression classification head
        self.expression_head = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_expression_classes)
        )
        
        # 2. Valence regression head
        self.valence_head = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

        # 3. Arousal regression head
        self.arousal_head = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        """Defines the forward pass of the model."""
        # 1. Get feature map from the DenseNet backbone
        # The .features attribute contains all the convolutional layers
        features = self.densenet.features(x)
        
        # 2. Perform global average pooling on the feature map
        out = nn.functional.relu(features, inplace=True)
        out = nn.functional.adaptive_avg_pool2d(out, (1, 1))
        
        # 3. Flatten the features to a vector
        features_flat = torch.flatten(out, 1)
        
        # 4. Pass the flattened features through each independent head
        expression_output = self.expression_head(features_flat)
        valence_output = self.valence_head(features_flat)
        arousal_output = self.arousal_head(features_flat)
        
        # 5. Return the outputs in a dictionary
        outputs = {
            'expression': expression_output,
            'valence': valence_output.squeeze(1),
            'arousal': arousal_output.squeeze(1)
        }
        
        return outputs

In [27]:
# --- Configuration Section ---
PROCESSED_DATA_DIR = '/kaggle/input/processed-data/processed_data'
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 15 
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {DEVICE}")

def main():
    # --- 1. Create Datasets and DataLoaders (This part is unchanged) ---
    train_dataset = FacialExpressionDataset(os.path.join(PROCESSED_DATA_DIR, 'train'))
    val_dataset = FacialExpressionDataset(os.path.join(PROCESSED_DATA_DIR, 'validation'))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    # --- 2. Initialize Model, Loss Functions, and Optimizer ---
    # === CHANGE 2: Initialize the DenseNet121 model ===
    model = MultiTaskDenseNet121(num_expression_classes=8).to(DEVICE)
    
    criterion_expression = nn.CrossEntropyLoss()
    criterion_valence_arousal = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # --- 3. Training and Validation Loop (This part is unchanged) ---
    for epoch in range(NUM_EPOCHS):
        print(f"\\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
        
        model.train()
        total_train_loss = 0.0
        for images, labels in tqdm(train_loader, desc="Training"):
            images = images.to(DEVICE)
            exp_labels, val_labels, aro_labels = labels['expression'].to(DEVICE), labels['valence'].to(DEVICE), labels['arousal'].to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss_exp = criterion_expression(outputs['expression'], exp_labels)
            loss_val = criterion_valence_arousal(outputs['valence'], val_labels)
            loss_aro = criterion_valence_arousal(outputs['arousal'], aro_labels)
            total_loss = loss_exp + loss_val + loss_aro
            
            total_loss.backward()
            optimizer.step()
            total_train_loss += total_loss.item()
            
        avg_train_loss = total_train_loss / len(train_loader)
        print(f"Average Training Loss: {avg_train_loss:.4f}")
        
        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation"):
                images = images.to(DEVICE)
                exp_labels, val_labels, aro_labels = labels['expression'].to(DEVICE), labels['valence'].to(DEVICE), labels['arousal'].to(DEVICE)
                outputs = model(images)
                loss_exp = criterion_expression(outputs['expression'], exp_labels)
                loss_val = criterion_valence_arousal(outputs['valence'], val_labels)
                loss_aro = criterion_valence_arousal(outputs['arousal'], aro_labels)
                total_loss = loss_exp + loss_val + loss_aro
                total_val_loss += total_loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        print(f"Average Validation Loss: {avg_val_loss:.4f}")

    print("\\nTraining complete.")
    # === CHANGE 3: Save the model with a new, descriptive name ===
    torch.save(model.state_dict(), 'multitask_densenet121_final.pth')
    print("Model saved to multitask_densenet121_final.pth")

if __name__ == '__main__':
    main()

Using device: cuda
Initialized dataset from '/kaggle/input/processed-data/processed_data/train'. Found 11196 samples.
Initialized dataset from '/kaggle/input/processed-data/processed_data/validation'. Found 600 samples.


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 162MB/s]


\n--- Epoch 1/15 ---


Training: 100%|██████████| 350/350 [01:50<00:00,  3.15it/s]


Average Training Loss: 1.9142


...

Average Validation Loss: 3.1208
\n--- Epoch 15/15 ---


Training: 100%|██████████| 350/350 [01:51<00:00,  3.15it/s]


Average Training Loss: 0.1015


Validation: 100%|██████████| 19/19 [00:02<00:00,  8.73it/s]


Average Validation Loss: 3.2734
\nTraining complete.
Model saved to multitask_densenet121_final.pth


In [28]:
# --- Configuration for DenseNet121 Evaluation ---

### CHANGE 1: Update the path to the DenseNet121 model's saved weights ###
MODEL_PATH = 'multitask_densenet121_final.pth' 

# These can stay the same
PROCESSED_DATA_DIR = '/kaggle/input/processed-data/processed_data' # Or your correct Kaggle path
BATCH_SIZE = 32
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Evaluating DenseNet121 on device: {DEVICE}")

# --- Custom Metric Implementations (This part is unchanged) ---
def sign_agreement_metric(y_true, y_pred):
    # ... (function code is the same)
    sagr = np.mean(np.sign(y_true) == np.sign(y_pred)) * 100
    return sagr

def concordance_correlation_coefficient(y_true, y_pred):
    # ... (function code is the same)
    cor = np.corrcoef(y_true, y_pred)[0, 1]
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true, var_pred = np.var(y_true), np.var(y_pred)
    sd_true, sd_pred = np.std(y_true), np.std(y_pred)
    numerator = 2 * cor * sd_true * sd_pred
    denominator = var_true + var_pred + (mean_true - mean_pred)**2
    return numerator / denominator

def evaluate_densenet_model(): # Renamed main function to avoid conflict
    # --- 1. Load Test Dataset (Unchanged) ---
    test_dataset = FacialExpressionDataset(os.path.join(PROCESSED_DATA_DIR, 'test'))
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # --- 2. Load Trained Model ---
    
    ### CHANGE 2: Initialize the DenseNet121 model architecture ###
    model = MultiTaskDenseNet121(num_expression_classes=8)
    
    # Load the state dict into the correct model architecture
    model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device(DEVICE)))
    model.to(DEVICE)
    model.eval()

    # --- 3. Get Predictions for the Entire Test Set (Unchanged) ---
    all_exp_preds, all_exp_labels = [], []
    all_val_preds, all_val_labels = [], []
    all_aro_preds, all_aro_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating DenseNet121"):
            images = images.to(DEVICE)
            outputs = model(images)
            
            exp_preds = torch.argmax(outputs['expression'], dim=1)
            all_exp_preds.extend(exp_preds.cpu().numpy())
            all_exp_labels.extend(labels['expression'].cpu().numpy())

            all_val_preds.extend(outputs['valence'].cpu().numpy())
            all_val_labels.extend(labels['valence'].cpu().numpy())
            
            all_aro_preds.extend(outputs['arousal'].cpu().numpy())
            all_aro_labels.extend(labels['arousal'].cpu().numpy())

    all_exp_preds, all_exp_labels = np.array(all_exp_preds), np.array(all_exp_labels)
    all_val_preds, all_val_labels = np.array(all_val_preds), np.array(all_val_labels)
    all_aro_preds, all_aro_labels = np.array(all_aro_preds), np.array(all_aro_labels)

    # --- 4. Calculate and Print Metrics (Unchanged, but now for DenseNet) ---
    print("\n" + "="*50)
    print("      DENSENET121 - EVALUATION RESULTS ON TEST SET")
    print("="*50 + "\n")

    # ... (The rest of the printing logic is exactly the same)
    print("--- Facial Expression (Categorical) ---")
    accuracy = accuracy_score(all_exp_labels, all_exp_preds)
    f1 = f1_score(all_exp_labels, all_exp_preds, average='weighted') 
    kappa = cohen_kappa_score(all_exp_labels, all_exp_preds)
    print(f"  Accuracy:       {accuracy:.4f}")
    print(f"  F1-Score:       {f1:.4f}")
    print(f"  Cohen's Kappa:  {kappa:.4f}\n")

    print("--- Valence (Continuous) ---")
    rmse_val = np.sqrt(mean_squared_error(all_val_labels, all_val_preds))
    corr_val, _ = pearsonr(all_val_labels, all_val_preds)
    sagr_val = sign_agreement_metric(all_val_labels, all_val_preds)
    ccc_val = concordance_correlation_coefficient(all_val_labels, all_val_preds)
    print(f"  Root Mean Square Error (RMSE): {rmse_val:.4f}")
    print(f"  Pearson Correlation (CORR):    {corr_val:.4f}")
    print(f"  Sign Agreement (SAGR):         {sagr_val:.2f}%")
    print(f"  Concordance Corr Coeff (CCC):  {ccc_val:.4f}\n")

    print("--- Arousal (Continuous) ---")
    rmse_aro = np.sqrt(mean_squared_error(all_aro_labels, all_aro_preds))
    corr_aro, _ = pearsonr(all_aro_labels, all_aro_preds)
    sagr_aro = sign_agreement_metric(all_aro_labels, all_aro_preds)
    ccc_aro = concordance_correlation_coefficient(all_aro_labels, all_aro_preds)
    print(f"  Root Mean Square Error (RMSE): {rmse_aro:.4f}")
    print(f"  Pearson Correlation (CORR):    {corr_aro:.4f}")
    print(f"  Sign Agreement (SAGR):         {sagr_aro:.2f}%")
    print(f"  Concordance Corr Coeff (CCC):  {ccc_aro:.4f}\n")
    print("="*50)

### CHANGE 3: Call the correct function ###
evaluate_densenet_model()

Evaluating DenseNet121 on device: cuda
Initialized dataset from '/kaggle/input/processed-data/processed_data/test'. Found 600 samples.


Evaluating DenseNet121: 100%|██████████| 19/19 [00:18<00:00,  1.02it/s]


      DENSENET121 - EVALUATION RESULTS ON TEST SET

--- Facial Expression (Categorical) ---
  Accuracy:       0.4650
  F1-Score:       0.4554
  Cohen's Kappa:  0.3878

--- Valence (Continuous) ---
  Root Mean Square Error (RMSE): 0.3861
  Pearson Correlation (CORR):    0.5933
  Sign Agreement (SAGR):         78.17%
  Concordance Corr Coeff (CCC):  0.5727

--- Arousal (Continuous) ---
  Root Mean Square Error (RMSE): 0.3554
  Pearson Correlation (CORR):    0.4654
  Sign Agreement (SAGR):         80.50%
  Concordance Corr Coeff (CCC):  0.4434



# VGG16

In [29]:
class MultiTaskVGG16(nn.Module):
    """
    A pre-trained VGG16 model adapted for our multi-task problem.
    VGG is a classic, deep CNN architecture known for its simplicity and power.
    """
    def __init__(self, num_expression_classes=8):
        super(MultiTaskVGG16, self).__init__()
        
        # Load a VGG16 model pre-trained on ImageNet
        self.vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        
        # Get the number of input features for the classifier
        # VGG's classifier input size is 512 * 7 * 7 = 25088
        num_ftrs = self.vgg.classifier[0].in_features
        
        # Replace the final classifier with an identity layer to extract features
        self.vgg.classifier = nn.Identity()

        # --- Define the three separate output heads ---
        
        self.expression_head = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_expression_classes)
        )
        
        self.valence_head = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

        self.arousal_head = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        """Defines the forward pass of the model."""
        # 1. Get feature map from the VGG backbone (.features)
        features = self.vgg.features(x)
        
        # 2. Flatten the feature map into a vector
        features_flat = torch.flatten(features, 1)
        
        # 3. Pass the flattened features through each independent head
        expression_output = self.expression_head(features_flat)
        valence_output = self.valence_head(features_flat)
        arousal_output = self.arousal_head(features_flat)
        
        outputs = {
            'expression': expression_output,
            'valence': valence_output.squeeze(1),
            'arousal': arousal_output.squeeze(1)
        }
        
        return outputs

In [30]:
# --- Configuration Section for VGG16---
PROCESSED_DATA_DIR = '/kaggle/input/processed-data/processed_data'
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 15 
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Starting VGG16 training on device: {DEVICE}")

def train_vgg_model(): # Renamed main function
    # --- 1. Create Datasets and DataLoaders (Unchanged) ---
    train_dataset = FacialExpressionDataset(os.path.join(PROCESSED_DATA_DIR, 'train'))
    val_dataset = FacialExpressionDataset(os.path.join(PROCESSED_DATA_DIR, 'validation'))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # --- 2. Initialize Model, Loss Functions, and Optimizer ---
    ### CHANGE 1: Initialize the VGG16 model ###
    model = MultiTaskVGG16(num_expression_classes=8).to(DEVICE)
    
    criterion_expression = nn.CrossEntropyLoss()
    criterion_valence_arousal = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # --- 3. Training and Validation Loop (Unchanged) ---
    for epoch in range(NUM_EPOCHS):
        print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
        
        model.train()
        total_train_loss = 0.0
        for images, labels in tqdm(train_loader, desc="Training"):
            images, exp_labels, val_labels, aro_labels = images.to(DEVICE), labels['expression'].to(DEVICE), labels['valence'].to(DEVICE), labels['arousal'].to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss_exp = criterion_expression(outputs['expression'], exp_labels)
            loss_val = criterion_valence_arousal(outputs['valence'], val_labels)
            loss_aro = criterion_valence_arousal(outputs['arousal'], aro_labels)
            total_loss = loss_exp + loss_val + loss_aro
            
            total_loss.backward()
            optimizer.step()
            total_train_loss += total_loss.item()
            
        avg_train_loss = total_train_loss / len(train_loader)
        print(f"Average Training Loss: {avg_train_loss:.4f}")
        
        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation"):
                images, exp_labels, val_labels, aro_labels = images.to(DEVICE), labels['expression'].to(DEVICE), labels['valence'].to(DEVICE), labels['arousal'].to(DEVICE)
                outputs = model(images)
                total_loss = criterion_expression(outputs['expression'], exp_labels) + criterion_valence_arousal(outputs['valence'], val_labels) + criterion_valence_arousal(outputs['arousal'], aro_labels)
                total_val_loss += total_loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        print(f"Average Validation Loss: {avg_val_loss:.4f}")

    print("\nTraining complete.")
    ### CHANGE 2: Save the model with a new name ###
    torch.save(model.state_dict(), 'multitask_vgg16_final.pth')
    print("Model saved to multitask_vgg16_final.pth")

### CHANGE 3: Call the training function ###
train_vgg_model()

Starting VGG16 training on device: cuda
Initialized dataset from '/kaggle/input/processed-data/processed_data/train'. Found 11196 samples.
Initialized dataset from '/kaggle/input/processed-data/processed_data/validation'. Found 600 samples.


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 193MB/s]  



--- Epoch 1/15 ---


Training: 100%|██████████| 350/350 [02:38<00:00,  2.20it/s]


Average Training Loss: 2.1036


...

Average Validation Loss: 3.7704

--- Epoch 15/15 ---


Training: 100%|██████████| 350/350 [02:38<00:00,  2.21it/s]


Average Training Loss: 0.0734


Validation: 100%|██████████| 19/19 [00:03<00:00,  5.75it/s]


Average Validation Loss: 3.9187

Training complete.
Model saved to multitask_vgg16_final.pth


In [32]:
# --- Configuration for DenseNet121 Evaluation ---

### CHANGE 1: Update the path to the DenseNet121 model's saved weights ###
MODEL_PATH = 'multitask_vgg16_final.pth' 

# These can stay the same
PROCESSED_DATA_DIR = '/kaggle/input/processed-data/processed_data' # Or your correct Kaggle path
BATCH_SIZE = 32
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Evaluating DenseNet121 on device: {DEVICE}")

# --- Custom Metric Implementations (This part is unchanged) ---
def sign_agreement_metric(y_true, y_pred):
    # ... (function code is the same)
    sagr = np.mean(np.sign(y_true) == np.sign(y_pred)) * 100
    return sagr

def concordance_correlation_coefficient(y_true, y_pred):
    # ... (function code is the same)
    cor = np.corrcoef(y_true, y_pred)[0, 1]
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true, var_pred = np.var(y_true), np.var(y_pred)
    sd_true, sd_pred = np.std(y_true), np.std(y_pred)
    numerator = 2 * cor * sd_true * sd_pred
    denominator = var_true + var_pred + (mean_true - mean_pred)**2
    return numerator / denominator

def evaluate_vgg16_model(): # Renamed main function to avoid conflict
    # --- 1. Load Test Dataset (Unchanged) ---
    test_dataset = FacialExpressionDataset(os.path.join(PROCESSED_DATA_DIR, 'test'))
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # --- 2. Load Trained Model ---
    
    ### CHANGE 2: Initialize the DenseNet121 model architecture ###
    model = MultiTaskVGG16(num_expression_classes=8)
    
    # Load the state dict into the correct model architecture
    model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device(DEVICE)))
    model.to(DEVICE)
    model.eval()

    # --- 3. Get Predictions for the Entire Test Set (Unchanged) ---
    all_exp_preds, all_exp_labels = [], []
    all_val_preds, all_val_labels = [], []
    all_aro_preds, all_aro_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating VGG16"):
            images = images.to(DEVICE)
            outputs = model(images)
            
            exp_preds = torch.argmax(outputs['expression'], dim=1)
            all_exp_preds.extend(exp_preds.cpu().numpy())
            all_exp_labels.extend(labels['expression'].cpu().numpy())

            all_val_preds.extend(outputs['valence'].cpu().numpy())
            all_val_labels.extend(labels['valence'].cpu().numpy())
            
            all_aro_preds.extend(outputs['arousal'].cpu().numpy())
            all_aro_labels.extend(labels['arousal'].cpu().numpy())

    all_exp_preds, all_exp_labels = np.array(all_exp_preds), np.array(all_exp_labels)
    all_val_preds, all_val_labels = np.array(all_val_preds), np.array(all_val_labels)
    all_aro_preds, all_aro_labels = np.array(all_aro_preds), np.array(all_aro_labels)

    # --- 4. Calculate and Print Metrics (Unchanged, but now for DenseNet) ---
    print("\n" + "="*50)
    print("      VGG16 - EVALUATION RESULTS ON TEST SET")
    print("="*50 + "\n")

    # ... (The rest of the printing logic is exactly the same)
    print("--- Facial Expression (Categorical) ---")
    accuracy = accuracy_score(all_exp_labels, all_exp_preds)
    f1 = f1_score(all_exp_labels, all_exp_preds, average='weighted') 
    kappa = cohen_kappa_score(all_exp_labels, all_exp_preds)
    print(f"  Accuracy:       {accuracy:.4f}")
    print(f"  F1-Score:       {f1:.4f}")
    print(f"  Cohen's Kappa:  {kappa:.4f}\n")

    print("--- Valence (Continuous) ---")
    rmse_val = np.sqrt(mean_squared_error(all_val_labels, all_val_preds))
    corr_val, _ = pearsonr(all_val_labels, all_val_preds)
    sagr_val = sign_agreement_metric(all_val_labels, all_val_preds)
    ccc_val = concordance_correlation_coefficient(all_val_labels, all_val_preds)
    print(f"  Root Mean Square Error (RMSE): {rmse_val:.4f}")
    print(f"  Pearson Correlation (CORR):    {corr_val:.4f}")
    print(f"  Sign Agreement (SAGR):         {sagr_val:.2f}%")
    print(f"  Concordance Corr Coeff (CCC):  {ccc_val:.4f}\n")

    print("--- Arousal (Continuous) ---")
    rmse_aro = np.sqrt(mean_squared_error(all_aro_labels, all_aro_preds))
    corr_aro, _ = pearsonr(all_aro_labels, all_aro_preds)
    sagr_aro = sign_agreement_metric(all_aro_labels, all_aro_preds)
    ccc_aro = concordance_correlation_coefficient(all_aro_labels, all_aro_preds)
    print(f"  Root Mean Square Error (RMSE): {rmse_aro:.4f}")
    print(f"  Pearson Correlation (CORR):    {corr_aro:.4f}")
    print(f"  Sign Agreement (SAGR):         {sagr_aro:.2f}%")
    print(f"  Concordance Corr Coeff (CCC):  {ccc_aro:.4f}\n")
    print("="*50)

### CHANGE 3: Call the correct function ###
evaluate_vgg16_model()

Evaluating DenseNet121 on device: cuda
Initialized dataset from '/kaggle/input/processed-data/processed_data/test'. Found 600 samples.


Evaluating VGG16: 100%|██████████| 19/19 [00:07<00:00,  2.66it/s]


      VGG16 - EVALUATION RESULTS ON TEST SET

--- Facial Expression (Categorical) ---
  Accuracy:       0.4417
  F1-Score:       0.4444
  Cohen's Kappa:  0.3624

--- Valence (Continuous) ---
  Root Mean Square Error (RMSE): 0.3859
  Pearson Correlation (CORR):    0.5492
  Sign Agreement (SAGR):         78.33%
  Concordance Corr Coeff (CCC):  0.4811

--- Arousal (Continuous) ---
  Root Mean Square Error (RMSE): 0.3693
  Pearson Correlation (CORR):    0.3981
  Sign Agreement (SAGR):         77.50%
  Concordance Corr Coeff (CCC):  0.3697

